This script will first prepare a new SQLite database to hold the KSP, KYTC API and other reference data.  Next it will consolidate the annual KSP datasets into a single pandas dataframe and then create a new table within the database for later visualization use. Finally, it will load several reference datasets needed for visualization.


In [ ]:
# Import modules
import os
from os.path import exists
import pandas as pd
import sqlite3

In [ ]:
# Define the path for the SQLite database
cwd = os.getcwd()
database_path = f'{cwd}/data/crash_data.db'
if exists(database_path):
    print("Databases already exists")
else:
    os.makedirs(os.path.dirname(database_path), exist_ok=True)

# Define the path for the raw crash data files downloaded
directory_path = f'{cwd}/data/raw_crash_data'

# Create/Connect to SQLite database
conn = sqlite3.connect(database_path)
cursor = conn.cursor()

In [ ]:
# Function to check if tables created in the database exist
def check_table_exists(db_path, table_name):
    # Connect to the SQLite database
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Query to check if the table exists
    cursor.execute('''
        SELECT name
        FROM sqlite_master
        WHERE type='table' AND name=?
    ''', (table_name,))

    # Fetch one record
    table_exists = cursor.fetchone() is not None

    # Close the connection
    conn.close()

    return table_exists

In [ ]:
# Function to check if tables created in the database have data
def check_table_has_data(db_path, table_name):
    # Connect to the SQLite database
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Query to count the number of rows in the table
    cursor.execute(f'SELECT COUNT(*) FROM {table_name}')

    # Fetch the count
    row_count = cursor.fetchone()[0]

    # Close the connection
    conn.close()

    return row_count > 0


The following section was created to generate the list for the table creation step.  It is not necessary to run it each time.

In [ ]:
# temp section to read the column names to create table columns
def read_column_names(csv_file_path):

    # Read only the first row of the CSV to get the column names
    df = pd.read_csv(csv_file_path, nrows=0)
    column_names = df.columns.tolist()
    return column_names

# Read the column names from the CSV fil- change as needed for different datasets
csv_file_path = f'{cwd}/data/raw_crash_data/Incidents_2024.csv'
# csv_file_path = f'{cwd}/data/raw_crash_data/IncidentTrafficControl_2024.csv'
# csv_file_path = f'{cwd}/data/raw_crash_data/Person_2024.csv'
# csv_file_path = f'{cwd}/data/raw_crash_data/Vehicle_2024.csv'

columns = read_column_names(csv_file_path)
print(columns)

The following cells check if the table exists in the SQLite database - this is a temporary check and will be commented out when tested fully.

In [ ]:
# Check to see if tables created exist
#table_name = 'ksp_incidents'
#table_name = 'ksp_controls'
#table_name = 'ksp_person'
table_name = 'ksp_vehicles'

if check_table_exists(database_path, table_name):
    print(f"The table '{table_name}' exists.")
else:
    print(f"The table '{table_name}' does not exist.")

In [ ]:
# Check of tables created are empty
table_name = 'ksp_incidents'
# table_name = 'ksp_controls'
# table_name = 'ksp_person'
# table_name = 'ksp_vehicles'

if check_table_has_data(database_path, table_name):
    print(f"The table '{table_name}' contains data.")
else:
    print(f"The table '{table_name}' is empty.")

In [ ]:
# Create collision incidents table in database
cursor.execute('''CREATE TABLE IF NOT EXISTS ksp_incidents (
        IncidentID int,
        AgencyORI int,
        AgencyName TEXT,
        IncidentStatusDesc TEXT,
        County TEXT,
        RdwyNumber TEXT,
        Street TEXT,
        RoadwayName TEXT,
        StreetSfx TEXT,
        StreetDir TEXT,
        IntersectionRdwy TEXT,
        IntersectionRdwyName TEXT,
        BetweenStRdwy1 TEXT,
        BetweenStRdwyName1 TEXT,
        BetweenStRdwy2 TEXT,
        BetweenStRdwyName2 TEXT,
        Latitude REAL,
        Longitude REAL,
        Milepoint REAL,
        CollisionDate DATE,
        CollisionTime TIME,
        UnitsInvolved INT,
        MotorVehiclesInvolved INT,
        NumberKilled INT,
        NumberInjured INT,
        Weather TEXT,
        RdwyConditionCode INT,
        HitandRun TEXT,
        DirAnalysisCode	TEXT,
        MannerofCollision TEXT,
        RdwyCharacter TEXT,
        LightCondition TEXT,
        RampFromRdwyId TEXT,
        RampToRdwyId TEXT,
        AcceptedDate DATE,
        IsSecondaryCollision TEXT,
        OwnerBadge TEXT,
        IncidentStatus TEXT);''')

In [ ]:
# Append collision_incidents to dataframe
csv_files = [f for f in os.listdir(directory_path) if f.startswith("Incidents_") and f.endswith(".csv")]

# Initialize an empty list to hold dataframes
dataframes = []

# Iterate through the CSV files and load them into dataframes
for csv_file in csv_files:
    file_path = os.path.join(directory_path, csv_file)
    df = pd.read_csv(file_path)
    dataframes.append(df)

# Concatenate all dataframes into a single dataframe
combined_incidents_df = pd.concat(dataframes, ignore_index=True)

# Drop any column with "Unnamed" in its name
unnamed_columns = [col for col in combined_incidents_df.columns if col.startswith('Unnamed')]
combined_incidents_df = combined_incidents_df.drop(columns=unnamed_columns)

# Standardize the date column
combined_incidents_df['StandardizedCollisionDate'] = pd.to_datetime(
    combined_incidents_df['CollisionDate'], errors='coerce').dt.strftime('%Y-%m-%d')

# Drop the original CollisionDate column if needed
combined_incidents_df = combined_incidents_df.drop(columns=['CollisionDate'])

# Rename the new column to CollisionDate if necessary
combined_incidents_df = combined_incidents_df.rename(
    columns={'StandardizedCollisionDate': 'CollisionDate'})

# Set the option to display all columns
pd.set_option('display.max_columns', None)

print("All CSV files have been successfully loaded into a single DataFrame.")
print(combined_incidents_df)

In [ ]:
# Specify the path where you want to save the CSV file
output_path = f'{cwd}/data/clean_crash_data/collision_incidents.csv'

# Export the dataframe to a CSV file
combined_incidents_df.to_csv(output_path, index=False)
print(f"Dataframe exported successfully to {output_path}")

In [ ]:
# Write the DataFrame to the SQLite table
combined_incidents_df.to_sql('ksp_incidents', conn, if_exists='append', index=False)

# Commit the changes and close the connection
conn.commit()
#conn.close() - only for the last dataset

print(f"Data from {df} has been successfully inserted into the collision_incidents table.")

The next dataset will be the collision_traffic_controls

In [ ]:
# Create table incident_traffic_control in database
cursor.execute('''CREATE TABLE IF NOT EXISTS ksp_controls (
        IncidentID int,
        TrafficControlNo int,
        TrafficControl TEXT);''')

In [ ]:

# Combine all CSV files in the directory that begin with "IncidentTrafficControl_"
csv_files = [f for f in os.listdir(directory_path) if f.startswith("IncidentTraffic") and f.endswith(".csv")]

# Initialize an empty list to hold dataframes
dataframes = []

# Iterate through the CSV files and load them into dataframes
for csv_file in csv_files:
    file_path = os.path.join(directory_path, csv_file)
    df = pd.read_csv(file_path)
    dataframes.append(df)

# Concatenate all dataframes into a single dataframe
combined_controls_df = pd.concat(dataframes, ignore_index=True)

#  Drop any column with "Unnamed" in its name
unnamed_columns = [col for col in combined_controls_df.columns if col.startswith('Unnamed')]
combined_controls_df = combined_controls_df.drop(columns=unnamed_columns)

# Set the option to display all columns
pd.set_option('display.max_columns', None)

print("All CSV files have been successfully loaded into a single DataFrame.")
print(combined_controls_df)

In [ ]:

# Specify the path where you want to save the CSV file
output_path = f'{cwd}/data/clean_crash_data/incident_traffic_controls.csv'

# Export the dataframe to a CSV file
combined_controls_df.to_csv(output_path, index=False)

print(f"Dataframe exported successfully to {output_path}")

In [ ]:
# Prepare the SQL insert statement dynamically based on DataFrame columns
columns = ', '.join([f'"{col}"' for col in combined_controls_df.columns])

placeholders = ', '.join(['?'] * len(combined_controls_df.columns))
sql = f'INSERT INTO ksp_controls ({columns}) VALUES ({placeholders})'

# Convert DataFrame to list of tuples
data_to_insert = combined_controls_df.to_records(index=False)

# Execute the SQL command using executemany
cursor.executemany(sql, data_to_insert)

# Commit changes and close the connection
conn.commit()

print("Data successfully added to the SQLite database at", database_path)

In [ ]:
# Write the DataFrame to the SQLite table
combined_controls_df.to_sql('ksp_controls', conn, if_exists='append', index=False)

# Commit the changes and close the connection
conn.commit()

print(f"Data from {df} has been successfully inserted into the collision_incidents table.")

The next cells are for the incident_vehicle dataset

In [ ]:
# Create table incident_vehicles in database
cursor.execute('''CREATE TABLE IF NOT EXISTS ksp_vehicles (
        IncidentID INT,
        UnitNumber INT,
        UnitType TEXT,
        AirbagSwitchCde TEXT,
        IsCommercialVeh TEXT,
        CrashAvoidCde TEXT,
        DriverIdentifiedCde TEXT,
        EventCollWithFirstCde TEXT,
        EventCollWithSecondCde TEXT,
        HasFire TEXT,
        PreCollActionCde TEXT,
        UnderOverrideCde TEXT,
        VehicleIsInsured TEXT,
        MakeCde TEXT,
        ModelCde TEXT,
        VehicleType TEXT,
        MakeDescription TEXT,
        ModelDescription TEXT);''')

In [ ]:

# Combine all CSV files in the directory that begin with "IncidentTrafficControl_"
csv_files = [f for f in os.listdir(directory_path)  if f.startswith("Vehicles_") and f.endswith(".csv")]

# Initialize an empty list to hold dataframes
dataframes = []

# Iterate through the CSV files and load them into dataframes
for csv_file in csv_files:
    file_path = os.path.join(directory_path, csv_file)
    df = pd.read_csv(file_path)
    dataframes.append(df)

# Concatenate all dataframes into a single dataframe
combined_vehicles_df = pd.concat(dataframes, ignore_index=True)

# Drop any column with "Unnamed" in its name
unnamed_columns = [col for col in combined_vehicles_df.columns if col.startswith('Unnamed')]
combined_vehicles_df = combined_vehicles_df.drop(columns=unnamed_columns)

# Set the option to display all columns
pd.set_option('display.max_columns', None)

print("All CSV files have been successfully loaded into a single DataFrame.")
print(combined_vehicles_df)

In [ ]:
# Specify the path where you want to save the CSV file
output_path = f'{cwd}/data/clean_crash_data/incident_vehicles.csv'

# Export the dataframe to a CSV file
combined_vehicles_df.to_csv(output_path, index=False)

print(f"Dataframe exported successfully to {output_path}")

In [ ]:
# Write the DataFrame to the SQLite table
combined_vehicles_df.to_sql('ksp_vehicles', conn, if_exists='append', index=False)

# Commit the changes and close the connection
conn.commit()

print(f"Data from {df} has been successfully inserted into the collision_incidents table.")

The following cells clean and load the ksp people csv files to the SQLite database.

In [ ]:
# Create table ksp_person in database
cursor.execute('''CREATE TABLE IF NOT EXISTS ksp_person (
        IncidentID INT,
        UnitNumber INT,
        PersonNo INT,
        PersonTypeCde TEXT,
        DeathDte DATE,
        AgeAtIncident INT,
        Gender TEXT,
        IsOwner TEXT,
        WasTransported TEXT,
        InjurySeverityCde TEXT,
        InjuryLocationCde TEXT,
        PosInVehicleCde TEXT,
        RestraintUseCde TEXT,
        TrappedCde TEXT,
        EjectionCde TEXT,
        EjectionPathCde TEXT,
        SuspectedOfDrinking TEXT,
        TestOffered TEXT,
        TestRefused TEXT,
        TestedForCde TEXT,
        TestSentTo TEXT,
        TestResults TEXT,
        HasOpLicense TEXT,
        HasCDLicense TEXT,
        HasLicenseRestrictions TEXT,
        HasOpEndorsements TEXT);''')

In [ ]:
# Combine all CSV files in the directory that begin with "IncidentTrafficControl_"
csv_files = [f for f in os.listdir(directory_path)  if f.startswith("Person_") and f.endswith(".csv")]
print(csv_files)

# Initialize an empty list to hold dataframes
dataframes = []

# Iterate through the CSV files and load them into dataframes
for csv_file in csv_files:
    file_path = os.path.join(directory_path, csv_file)
    df = pd.read_csv(file_path)
    dataframes.append(df)

# Concatenate all dataframes into a single dataframe
combined_person_df = pd.concat(dataframes, ignore_index=True)

# Drop any column with "Unnamed" in its name
#unnamed_columns = [col for col in combined_person_df.columns if col.startswith('Unnamed')]
#combined_person_df = combined_person_df.drop(columns=unnamed_columns)

# Set the option to display all columns
pd.set_option('display.max_columns', None)

print("All CSV files have been successfully loaded into a single DataFrame.")
print(combined_person_df)

In [ ]:
# Specify the path where you want to save the CSV file for ksp person data
output_path = f'{cwd}/data/clean_crash_data/ksp_person.csv'

# Export the dataframe to a CSV file
combined_person_df.to_csv(output_path, index=False)

print(f"Dataframe exported successfully to {output_path}")

In [ ]:
# Write the KSP_Person DataFrame to the SQLite table
combined_person_df.to_sql('ksp_person', conn, if_exists='append', index=False)

# Commit the changes and close the connection
conn.commit()

print(f"Data from {df} has been successfully inserted into the collision_incidents table.")

The following cells ingest the incident unit factor datasets

In [ ]:
# Create table incident_vehicles in database
cursor.execute('''CREATE TABLE IF NOT EXISTS ksp_factors (
        IncidentID INT,
        UnitNumber INT,
        Factor_Type TEXT,
        Factor TEXT);''')

In [ ]:
# Combine all CSV files in the directory that begin with "IncidentTrafficControl_"
csv_files = [f for f in os.listdir(directory_path)  if f.startswith("Unit_") and f.endswith(".csv")]

# Initialize an empty list to hold dataframes
dataframes = []

# Iterate through the CSV files and load them into dataframes
for csv_file in csv_files:
    file_path = os.path.join(directory_path, csv_file)
    df = pd.read_csv(file_path)
    dataframes.append(df)

# Concatenate all dataframes into a single dataframe
combined_factors_df = pd.concat(dataframes, ignore_index=True)

# Drop any column with "Unnamed" in its name
unnamed_columns = [col for col in combined_factors_df.columns if col.startswith('Unnamed')]
combined_factors_df = combined_factors_df.drop(columns=unnamed_columns)

# Set the option to display all columns
pd.set_option('display.max_columns', None)

print("All CSV files have been successfully loaded into a single DataFrame.")
print(combined_factors_df)

In [ ]:
# Specify the path where you want to save the CSV file
output_path = f'{cwd}/data/clean_crash_data/incident_factors.csv'

# Export the dataframe to a CSV file
combined_factors_df.to_csv(output_path, index=False)

print(f"Dataframe exported successfully to {output_path}")

In [ ]:

# Write the DataFrame to the SQLite table
combined_factors_df.to_sql('ksp_factors', conn, if_exists='append', index=False)

# Commit the changes and close the connection
conn.commit()

print(f"Data from {df} has been successfully inserted into the ksp_factors table.")

The following cells load Code Lookup tables into the database

In [ ]:
# Change directory path to reference data
# Define the path for the raw crash data files downloaded
directory_path = f'{cwd}/data/reference_data'

In [ ]:
# Create table incident_vehicles in database
cursor.execute('''CREATE TABLE IF NOT EXISTS county_district_lut (
        OBJECTID INT,
        Cnty_Name_UC TEXT,
        Cnty_Name_PC TEXT,
        Cnty_Number INT,
        Cnty_FIPS_Number INT,
        KYTC_District_Number INT,
        D_DISTRICT TEXT
        );''')

In [ ]:
csv_file = directory_path + '/county_lut.csv'
file_path = os.path.join(directory_path, csv_file)
df = pd.read_csv(file_path)

In [ ]:
# Write the DataFrame to the SQLite table
df.to_sql('county_district_lut', conn, if_exists='append', index=False)

# Commit the changes and close the connection
conn.commit()

print(f"Data from {df} has been successfully inserted into the county_district_lut table.")

In [ ]:
# Create table incident_Factors in database
cursor.execute('''CREATE TABLE IF NOT EXISTS unit_factor_code_lut (
        Factor_code TEXT,
        Description TEXT
        );''')

In [ ]:
csv_file = directory_path + '/factor_code_lut.csv'
file_path = os.path.join(directory_path, csv_file)
df = pd.read_csv(file_path)

In [ ]:
# Write the DataFrame to the SQLite table
df.to_sql('unit_factor_code_lut', conn, if_exists='append', index=False)

# Commit the changes and close the connection
conn.commit()
conn.close()

print(f"Data from {df} has been successfully inserted into the unit factor code lookup table.")